**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/peyrache-2015"], check=True)
    os.chdir("/content/encore/entries/peyrache-2015")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# The findings of peyrache-2015-01, checked on DANDI:000056 version 0.261106.1949

Attempt `peyrache-2015-01` tested Peyrache, Lacroix, Petersen and Buzsaki (2015), *Internally
organized mechanisms of the head direction sense* (Nature Neuroscience 18, 569-575,
doi:10.1038/nn.3968), on version 0.250624.0430 of dandiset 000056, and recorded ten findings about
the data and the paper. Attempt `peyrache-2015-02`, made blind on version 0.261106.1949, reports
some of them again; this notebook checks each earlier finding directly on version 0.261106.1949, in one
short section per finding, each opening with the earlier statement. It is part of the
reconciliation of the two attempts (docs/protocol.md, "Repeat attempts"). It tests no claim.

Data are streamed with the entry's helper `hd.py` (dandiset 000056, version 0.261106.1949);
nothing is written to disk.

In [ ]:
import json, re
import numpy as np
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
import hd

print("dandiset", hd.DANDISET, "version", hd.VERSION)
ASSETS = hd.list_assets()
print(len(ASSETS), "assets")

In [ ]:
def txt(x):
    return hd._txt(x)


def survey(asset):
    """One session: what hd.load_session reads, and the metadata the earlier findings are about."""
    path, asset_id, _ = asset
    s = hd.load_session(asset_id, path)
    f = hd.open_asset(asset_id)
    u = f["units"]
    s["col_desc"] = {k: txt(u[k].attrs.get("description", "")) for k in u.keys()
                     if k in ("in_lab_cell_list", "region", "shank", "cluster_id", "lratio", "is_hd_cell")}
    s["in_lab"] = u["in_lab_cell_list"][:].astype(bool) if "in_lab_cell_list" in u else None
    s["lratio"] = u["lratio"][:] if "lratio" in u else None
    el = f["general/extracellular_ephys/electrodes"]
    s["electrode_locations"] = sorted(set(txt(x) for x in el["location"][:]))
    s["acq"] = {}
    for k in f["acquisition"]:
        g = f["acquisition"][k]
        if "data" in g:
            d = g["data"]
            s["acq"][k] = dict(unit=txt(d.attrs.get("unit", "")), conversion=float(d.attrs.get("conversion", np.nan)),
                               max_first_1000=float(np.nanmax(d[:1000])) if np.isfinite(d[:1000]).any() else np.nan)
    s["pos_end"] = {}
    if "processing/behavior/position" in f:
        for k in f["processing/behavior/position"]:
            g = f["processing/behavior/position"][k]
            r = g["starting_time"].attrs["rate"]
            s["pos_end"][k] = float(g["starting_time"][()]) + g["data"].shape[0] / r
    s["rec_end"] = max((b for _, a, b in s["sub_recordings"]), default=np.nan)
    s["ecephys"] = {}
    if "processing/ecephys" in f:
        for k in f["processing/ecephys"]:
            g = f["processing/ecephys"][k]
            if txt(g.attrs.get("neurodata_type", "")) == "TimeIntervals":
                s["ecephys"][k] = dict(n=len(g["id"]), description=txt(g.attrs.get("description", "")),
                                       columns={c: txt(g[c].attrs.get("description", "")) for c in g.keys()
                                                if c != "id" and hasattr(g[c], "attrs")})
    s["acquisition_keys"] = sorted(f["acquisition"].keys())
    f.close()
    s["name"] = path.split("_ses-")[1].split("_")[0]
    return s


with ThreadPoolExecutor(4) as ex:
    S = list(ex.map(survey, ASSETS))
S = {s["name"]: s for s in S}
print(len(S), "sessions read")

In [ ]:
rows = []
for n, s in S.items():
    tags = sorted({t for e in s["epochs"] for t in e[0]})
    rows.append(dict(session=n, subject=s["subject"], units=len(s["spikes"]),
                     ADn=int((s["region"] == hd.ADN).sum()), PoS=int((s["region"] == hd.POS).sum()),
                     other=int(((s["region"] != hd.ADN) & (s["region"] != hd.POS)).sum()),
                     in_lab=int(s["in_lab"].sum()) if s["in_lab"] is not None else -1,
                     head_direction=s["hd"] is not None, sleep_states=len(s["states"]) > 0,
                     open_field="open_field" in tags, n_epochs=len(s["epochs"]),
                     lfp=s["lfp_shape"] is not None))
T = pd.DataFrame(rows).set_index("session")
pd.set_option("display.width", 200)
print(T.to_string())
print("\nBy subject:")
print(T.groupby("subject")[["units", "ADn", "PoS", "other", "in_lab"]].sum().to_string())
print("\nSessions", len(T), "; subjects", T.subject.nunique(), "; unit rows", int(T.units.sum()))
print("Unit columns (union over files):", sorted(set().union(*[s["columns"] for s in S.values()])))

## Earlier finding 1

> The units tables hold only spike times. No unit has an electrode, shank, brain region, waveform or quality measure, and every electrode's location is "unknown". Units cannot be assigned to the antero-dorsal thalamus or the post-subiculum, which blocks 13 of the paper's 21 claims. The electrode groups and ripple annotations are consistent with Mouse24, Mouse25 and Mouse28 being the three dual-site mice, but the files do not say so.

In [ ]:
cols = ["region", "electrode_group", "shank", "cluster_id", "lratio", "buzcode_mean_waveform", "lab_mean_waveform"]
have = {c: sum(c in s["columns"] for s in S.values()) for c in cols}
print("Files (of", len(S), ") whose units table has each column:", have)
reg = pd.Series(np.concatenate([s["region"] for s in S.values()])).value_counts()
print("Units by region over all files:"); print(reg.to_string())
locs = sorted(set().union(*[s["electrode_locations"] for s in S.values()]))
print("Electrode locations over all files:", locs)
print("Files where every electrode location is 'unknown':", sum(s["electrode_locations"] == ["unknown"] for s in S.values()))
both = T[(T.ADn > 0) & (T.PoS > 0)]
print("Sessions with units labelled in both structures, by subject:", both.groupby("subject").size().to_dict())

## Earlier finding 2

> The files hold 1959 unit rows (103 with no spikes) in 40 sessions, against the paper's 1077 recorded cells in 42 sessions. Mouse12-120809 alone has 292 units, 167 of them firing below 0.5 Hz. Unit ids run from 0 to N-1 and carry no shank or cluster number, the units table has no quality measure, and 1526 rows fire at 0.5 Hz or more, so the paper's 1077 cells cannot be recovered. Nothing marks which units or sessions the paper analysed (its Methods discard some sessions).

In [ ]:
n_empty = sum(int(sum(len(x) == 0 for x in s["spikes"])) for s in S.values())
print("Sessions", len(S), "; unit rows", int(T.units.sum()), "; rows with no spikes", n_empty)
print("Mouse12-120809:", int(T.loc["Mouse12-120809", "units"]), "units, by region",
      pd.Series(S["Mouse12-120809"]["region"]).value_counts().to_dict())
print("Files with shank and cluster_id:", sum(("shank" in s["columns"]) and ("cluster_id" in s["columns"]) for s in S.values()),
      "; with lratio:", sum(s["lratio"] is not None for s in S.values()))
lab = np.concatenate([s["in_lab"] for s in S.values()])
regs = np.concatenate([s["region"] for s in S.values()])
print("Units in the lab's cell list (in_lab_cell_list):", int(lab.sum()),
      "; of them ADn", int((lab & (regs == hd.ADN)).sum()), "and PoS", int((lab & (regs == hd.POS)).sum()),
      "(paper: 720 thalamic and 357 PoS cells, 1077 in all)")
print("Description of in_lab_cell_list:", S["Mouse12-120806"]["col_desc"]["in_lab_cell_list"])

## Earlier finding 3

> The open-field exploration (the paper's waking epoch) is not marked. The "Awake" state also covers waking in the home cage, and some sessions hold a second exploration or a larger second environment. The epoch had to be found from position.

In [ ]:
print("Sessions with an epochs table:", int((T.n_epochs > 0).sum()), "of", len(T),
      "; with an open_field epoch:", int(T.open_field.sum()))
print("Sessions without one:", T.index[~T.open_field].tolist())
tags = pd.Series([t for s in S.values() for e in s["epochs"] for t in e[0]]).value_counts()
print("Epoch tags over all files:", tags.to_dict())
# How much of the scored wake state lies in the open_field epochs
r = []
for n, s in S.items():
    of = np.array([(a, b) for tg, a, b in s["epochs"] if "open_field" in tg])
    if "wake" in s["states"] and len(of):
        w = s["states"]["wake"]
        ov = sum(max(0.0, min(b, y) - max(a, x)) for a, b in w for x, y in of)
        r.append(ov / hd.total(w))
r = np.array(r)
print(f"Fraction of the scored wake state inside open_field epochs, over {len(r)} sessions: "
      f"min {r.min():.3f}, median {np.median(r):.3f}, max {r.max():.3f}")

## Earlier finding 4

> In 36 of 40 files one unit's spike times are not sorted. Each such unit holds exactly two sorted runs, and in all 36 the first run ends at an index of the form k x 65536 - 1. The second run starts within 36 s of the start of the recording and the first between 24 and 13704 s, and the two end within 15 s of each other. In 26 units no time of the second run appears in the first; in 10, 821 to 22281 times do. This suggests an error in the conversion.

In [ ]:
uns = {n: [k for k, x in enumerate(s["spikes"]) if np.any(np.diff(x) < 0)] for n, s in S.items()}
print("Files with a unit whose spike times are not sorted:", sum(len(v) > 0 for v in uns.values()), "of", len(S))
print("Units with unsorted spike times:", sum(len(v) for v in uns.values()), "of", int(T.units.sum()))

## Earlier finding 5

> In Mouse12-120808 the position and the spikes do not share a clock. Head-direction tuning is best with position shifted by -1.5 s at the start of the exploration and -3.4 s at its end, and no unit passes the HD criteria as deposited; with position realigned by the best shift of each quarter, 13 units do. In all other sessions the best shift is -0.2 to 0 s.

In [ ]:
# As in the earlier attempt: the mean resultant length of the waking tuning curve of the six
# most tuned ADn units, with the head direction shifted against the spikes (a negative shift pairs
# each spike with the head direction that many seconds earlier), over the whole wake interval and
# in each of its quarters. The six units are chosen by their best resultant length over shifts of
# -3 to 0 s, so that a shifted clock cannot hide them.
SHIFTS = np.round(np.arange(-5, 1.01, 0.1), 2)


def rlen(tc):
    return float(np.abs(np.sum(tc * np.exp(1j * hd.ANG))) / np.sum(tc))


def shift_curve(s, iv, units):
    t, a = hd.wake_hd(s)
    occ, dt = hd.occupancy(a), 1 / s["hd_rate"]
    R = [[rlen(hd.tuning_curve(hd.spike_hd(hd.restrict(s["spikes"][k], iv) + sh, t, a, s["hd_rate"]), occ, dt))
          for sh in SHIFTS] for k in units]
    return np.mean(R, 0)


def top_units(s, n=6):
    t, a = hd.wake_hd(s)
    occ, dt = hd.occupancy(a), 1 / s["hd_rate"]
    w = s["states"]["wake"]
    adn = np.where(s["region"] == hd.ADN)[0]
    best = [max(rlen(hd.tuning_curve(hd.spike_hd(hd.restrict(s["spikes"][k], w) + sh, t, a, s["hd_rate"]), occ, dt))
                for sh in (-3, -2, -1, 0)) for k in adn]
    return adn[np.argsort(best)[::-1][:n]]


rows = []
for n, s in S.items():
    if s["hd"] is None or "wake" not in s["states"] or (s["region"] == hd.ADN).sum() < 6:
        continue
    units = top_units(s)
    R = shift_curve(s, s["states"]["wake"], units)
    rows.append(dict(session=n, best_shift_s=SHIFTS[R.argmax()], r_best=R.max(), r_at_0=R[SHIFTS == 0][0]))
B = pd.DataFrame(rows).set_index("session")
print(B.round(2).to_string())
other = B.drop(index="Mouse12-120808")
print(f"\nBest shift in the {len(other)} other sessions: {other.best_shift_s.min():+.1f} to {other.best_shift_s.max():+.1f} s")

s = S["Mouse12-120808"]
w = s["states"]["wake"]
print("\nMouse12-120808 wake interval(s):", np.round(w).astype(int).tolist())
units = top_units(s)
a0, a1 = w[0, 0], w[-1, 1]
for q in range(4):
    q0, q1 = a0 + q * (a1 - a0) / 4, a0 + (q + 1) * (a1 - a0) / 4
    iv = np.array([[q0, q1]])
    R = shift_curve(s, iv, units)
    print(f"  quarter {q + 1} ({q0:.0f} to {q1:.0f} s): best shift {SHIFTS[R.argmax()]:+.1f} s, r {R.max():.2f} (at 0 s {R[SHIFTS == 0][0]:.2f})")
print("Mouse12-120808 units the files mark as HD cells (is_hd_cell):", int(np.nansum(s["is_hd_cell"].astype(float))))

## Earlier finding 6

> The raw LED positions in `acquisition` are stated in meters with a conversion of NaN but hold camera pixels (values in the hundreds).

In [ ]:
acq = pd.DataFrame([dict(session=n, series=k, **v) for n, s in S.items() for k, v in s["acq"].items()])
print("Acquisition series names:", sorted(acq.series.unique()))
print("Units:", acq.unit.value_counts().to_dict(), "; conversions:", acq.conversion.value_counts(dropna=False).to_dict())
print(f"Largest value in the first 1000 samples: {acq.max_first_1000.min():.1f} to {acq.max_first_1000.max():.1f}")

## Earlier finding 7

> Mouse32-140820 has no sleep-state table and no processed position, and Mouse17-130201 has no processed position; both have raw LED series only. In three Mouse20 sessions the position series ends 11 to 109 minutes before the recording.

In [ ]:
for n in ("Mouse32-140820", "Mouse17-130201"):
    s = S[n]
    print(n, "; head direction:", s["hd"] is not None, "; processed position:", sorted(s["pos_end"]),
          "; sleep states:", sorted(s["states"]), "; acquisition:", s["acquisition_keys"])
print()
for n, s in S.items():
    if s["pos_end"]:
        gap = s["rec_end"] - max(s["pos_end"].values())
        if gap > 60:
            after = [(tg, round(a), round(b)) for tg, a, b in s["epochs"] if b > max(s["pos_end"].values()) + 1]
            print(f"{n}: position ends at {max(s['pos_end'].values()):.1f} s, the recording at {s['rec_end']:.1f} s "
                  f"({gap:.0f} s, {gap / 60:.0f} min, earlier); epochs after the end of position: {after}")

## Earlier finding 8

> The annotation series `dow`, `hdD` and `rip` (labels such as "DOWN ON 5", "HD UP ON", "Ripple peak 1") have no description.

In [ ]:
names = sorted(set().union(*[s["ecephys"].keys() for s in S.values()]))
print("TimeIntervals tables in processing/ecephys:", names)
for k in names:
    ss = [s for s in S.values() if k in s["ecephys"]]
    no_desc = sum(not s["ecephys"][k]["description"] for s in ss)
    no_col = sum(any(not v for v in s["ecephys"][k]["columns"].values()) for s in ss)
    print(f"\n{k}: in {len(ss)} files; without a description {no_desc}; with an undescribed column {no_col}")
    print("  description:", ss[0]["ecephys"][k]["description"][:300])
    print("  columns:", {c: v[:80] for c, v in ss[0]["ecephys"][k]["columns"].items()})
print("\nAcquisition or annotation series named dow, hdD or rip:",
      sorted({k for s in S.values() for k in s["acquisition_keys"] + list(s["ecephys"]) if k in ("dow", "hdD", "rip")}))

## Earlier finding 9

> The dandiset's related resource for the paper gives its name and identifier as "nternally organized mechanisms of the head direction sense" (first letter missing, and the title in place of a DOI); the URL is correct.

In [ ]:
meta = requests.get(f"{hd.API}/dandisets/{hd.DANDISET}/versions/{hd.VERSION}/", timeout=60).json()
print(json.dumps(meta.get("relatedResource"), indent=1))

## Earlier finding 10

> Neither the paper nor the dandiset links analysis code, so the von Mises fit, the Rayleigh test, the scaling in the cross-correlogram estimate of angular velocity and the coherence measure had to be chosen or could not be repeated.

In [ ]:
rr = meta.get("relatedResource", [])
print("Related resources:", [(r.get("relation"), r.get("resourceType"), r.get("url")) for r in rr])
code_like = [r for r in rr if r.get("resourceType") in ("dcite:Software", "dcite:ComputationalNotebook")
             or "github" in (r.get("url") or "")]
print("Related resources that are code:", code_like)
desc = meta.get("description", "")
print("Description mentions code or GitHub:", bool(re.search(r"github|code|software", desc, re.I)))
print("The paper itself is unchanged; its Statistical Analysis section says only that analyses were run in Matlab, "
      "and it links no code (claims/peyrache-2015.yaml, opening comment).")

## Summary

The status of each earlier finding, with the numbers above, is recorded in the `reconciliation`
block of `attempts/peyrache-2015-02.yaml`.